In [65]:
import warnings
warnings.filterwarnings("ignore")

In [66]:
from pathlib import Path

In [67]:
import numpy as np
import pandas as pd

In [68]:
from sentence_transformers import SentenceTransformer

In [69]:
import faiss

## Configuration

In [70]:
RANDOM_STATE = 42

PROCESSED_DIR = Path("../artifacts/processed")
FEATURE_DIR = Path("../artifacts/features")
INDEX_DIR = Path("../artifacts/indexes/faiss")
SEMANTIC_DIR = Path("../artifacts/semantic")

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True,exist_ok=True)
SEMANTIC_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = ("sentence-transformers/all-MiniLM-L6-v2")

TOP_K = 10

RETRIEVAL_K = 100

## Load Data

In [71]:
consumer = pd.read_parquet(PROCESSED_DIR / "consumer_clean.parquet")
content = pd.read_parquet(PROCESSED_DIR / "content_clean.parquet")
print("Consumer shape:", consumer.shape)
print("Content shape:", content.shape)

Consumer shape: (72312, 11)
Content shape: (3122, 16)


## Standardize Columns

In [72]:
consumer.columns = (consumer.columns.str.strip().str.lower())
content.columns = (content.columns.str.strip().str.lower())

## Convert Timestamps

In [73]:
consumer["event_datetime"] = pd.to_datetime(consumer["event_timestamp"],unit="s",utc=True)
content["content_event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True)

## 1. Determine Available Articles

In [74]:
content_sorted = (content.sort_values(["item_id","content_event_datetime"]))

latest_content_state = (content_sorted.groupby("item_id").tail(1).copy())

latest_content_state["is_available"] = (latest_content_state["interaction_type"].astype(str).str.lower().eq("content_present"))

In [75]:
available_items = set(latest_content_state.loc[latest_content_state["is_available"],"item_id"])

pulled_items = set(latest_content_state.loc[~latest_content_state["is_available"],"item_id"])

print("Available:", len(available_items))

print("Pulled:", len(pulled_items))

Available: 2983
Pulled: 74


## 2. English Only Content

In [76]:
semantic_articles = content[content["language"].astype(str).str.lower().eq("en")].copy()

semantic_articles = semantic_articles[semantic_articles["item_id"].isin(available_items)].copy()

semantic_articles = (semantic_articles.drop_duplicates(subset=["item_id"]).reset_index(drop=True))

print("Semantic articles:", len(semantic_articles))

Semantic articles: 2166


## 3. Clean Title And Description

In [77]:
semantic_articles["title"] = (semantic_articles["title"].fillna("").astype(str).str.strip())

semantic_articles["text_description"] = (semantic_articles["text_description"].fillna("").astype(str).str.strip())

## 4. Construct Semantic Text

In [78]:
semantic_articles["article_text"] = ("Title: " + semantic_articles["title"] + ". Description: " +semantic_articles["text_description"])

## 5. Remove Unusable Articles

In [79]:
semantic_articles = semantic_articles[semantic_articles["article_text"].str.replace("Title:", "",regex=False).str.replace("Description:", "",regex=False).str.strip().ne("")].copy()

semantic_articles = (semantic_articles.reset_index(drop=True))

## 6. Load Sentence Transformer

In [80]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model:", EMBEDDING_MODEL_NAME)

Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 7. Generate Article Embeddings

In [81]:
article_texts = (semantic_articles["article_text"].tolist())

article_embeddings = (embedding_model.encode(article_texts, normalize_embeddings=True,batch_size=64, show_progress_bar=True, convert_to_numpy=True))

print("Embedding shape:", article_embeddings.shape)

Batches: 100%|██████████| 34/34 [00:16<00:00,  2.11it/s]

Embedding shape: (2166, 384)


## 8. Verify Embedding Quality

In [82]:
print("Number of articles:", len(article_embeddings))

print("Embedding dimension:", article_embeddings.shape[1])

embedding_norms = np.linalg.norm(article_embeddings, axis=1)

print("Minimum norm:", embedding_norms.min())

print("Maximum norm:", embedding_norms.max())

Number of articles: 2166
Embedding dimension: 384
Minimum norm: 0.9999999
Maximum norm: 1.0000001


## 9. Save Embeddings

In [83]:
embedding_path = (FEATURE_DIR / "article_embeddings.npy")

np.save(embedding_path, article_embeddings)

print("Saved:", embedding_path)

Saved: ..\artifacts\features\article_embeddings.npy


## 10. Save Article Embedding Mapping

In [84]:
embedding_metadata = (semantic_articles[["item_id", "title", "language", "item_type","producer_id"]].copy())

embedding_metadata["embedding_index"] = np.arange(len(embedding_metadata))

embedding_metadata.to_parquet(FEATURE_DIR / "article_embedding_index.parquet", index=False)

print("Saved article embedding metadata.")

Saved article embedding metadata.


## 10. Bulid Item ID Lookup

In [85]:
item_to_embedding_index = {item_id: int(index) for item_id, index in zip(embedding_metadata["item_id"], embedding_metadata["embedding_index"])}

index_to_item = {int(index): item_id for item_id, index in zip(embedding_metadata["item_id"],embedding_metadata["embedding_index"])}

## 11. Bulid FAISS Index

In [86]:
embedding_dimension = (article_embeddings.shape[1])

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(article_embeddings.astype(np.float32))

print("FAISS index size:", faiss_index.ntotal)

print("Embedding dimension:", faiss_index.d)

FAISS index size: 2166
Embedding dimension: 384


## 12. Save FAISS Index

In [87]:
faiss_index_path = (INDEX_DIR / "article_cosine.index")

faiss.write_index(faiss_index, str(faiss_index_path))

print("Saved:", faiss_index_path)

Saved: ..\artifacts\indexes\faiss\article_cosine.index


## 13. Save Metadata Beside Index

In [88]:
embedding_metadata.to_parquet(INDEX_DIR / "article_metadata.parquet", index=False)

## 14. Search Similar Articles

In [89]:
def search_similar_articles(item_id, top_k=10):
    if item_id not in item_to_embedding_index:
        return pd.DataFrame(
            columns=["item_id", "title", "similarity"])
    embedding_idx = (item_to_embedding_index[item_id])
    query_vector = (article_embeddings[embedding_idx].reshape(1, -1).astype(np.float32))
    scores, indices = (faiss_index.search(query_vector, top_k + 1))
    rows = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        candidate_item = (index_to_item[int(idx)])
        if candidate_item == item_id:
            continue
        rows.append({"item_id": candidate_item, "similarity": float(score)})
    result = pd.DataFrame(rows)
    result = result.merge(embedding_metadata[["item_id", "title", "language", "item_type"]],on="item_id", how="left")
    return result.head(top_k)

## 15. Test Article Similartiy

In [90]:
example_item = (semantic_articles["item_id"].iloc[0])

similar_articles = (search_similar_articles(example_item,top_k=10))

display(similar_articles)

,item_id,similarity,title,language,item_type
0,3353902017498793780,0.739046,The Rise And Growth of Ethereum Gets Mainstrea...,en,HTML
1,5274322067107287523,0.687925,Ethereum and Bitcoin Are Market Leaders But No...,en,HTML
2,8084284001249507595,0.600990,Microsoft Continues to Embrace Ethereum & Bitc...,en,HTML
3,-9171475473795142532,0.587986,Decentralized Options Exchange Etheropt Uses A...,en,HTML
4,5410603488837817052,0.586980,Banks find blockchain hard to put into practic...,en,HTML
5,3067875254349597654,0.526950,Microsoft Adds Ethereum to Windows Platform Fo...,en,HTML
6,607684800821303652,0.511327,How This Former Google Engineer Is Bringing Bl...,en,HTML
7,-4917007328809735647,0.507044,"Google Failure, Ethereum Leaps, ECB Giveout in...",en,HTML
8,8550670510357310628,0.506871,Why Many Smart Contract Use Cases Are Simply I...,en,HTML
9,4849766494522371290,0.505180,Cashila Announces Convenient Buy and Sell Feat...,en,HTML


## 16. User History

In [91]:
consumer_sorted = (consumer.sort_values(["consumer_id", "event_datetime"]).copy())

user_histories = (consumer_sorted.groupby("consumer_id")["item_id"].apply(list).to_dict())

## 17. Interaction Weights

In [92]:
INTERACTION_WEIGHTS = {"content_watched": 1.0, "content_liked": 2.0, "content_saved": 3.0,"content_followed": 4.0, "content_commented_on": 5.0}

consumer["interaction_weight"] = (consumer["interaction_type"].map(INTERACTION_WEIGHTS).fillna(0.0))

## 18. Recency Weighted User Embeddings

In [93]:
RECENCY_HALF_LIFE_DAYS = 14

In [94]:
def calculate_recency_weight(event_time, reference_time):
    age_days = (reference_time - event_time).total_seconds() / 86400
    age_days = max(age_days, 0)
    return (0.5 ** (age_days / RECENCY_HALF_LIFE_DAYS))

## 19. Bulid User Profile Embedding

In [95]:
def build_user_embedding(user_id, reference_time=None):
    user_history = (consumer[consumer["consumer_id"].eq(user_id)].sort_values("event_datetime"))
    if user_history.empty:
        return None
    if reference_time is None:
        reference_time = (consumer["event_datetime"].max())
    weighted_vectors = []
    weights = []
    for _, row in user_history.iterrows():
        item_id = row["item_id"]
        if item_id not in item_to_embedding_index:
            continue
        embedding_idx = (item_to_embedding_index[item_id])
        interaction_weight = (float(row["interaction_weight"]))
        recency_weight = (calculate_recency_weight(row["event_datetime"],reference_time))
        final_weight = (interaction_weight * recency_weight)
        if final_weight <= 0:
            continue
        vector = (article_embeddings[embedding_idx])
        weighted_vectors.append(vector * final_weight)
        weights.append(final_weight)
    if not weighted_vectors:
        return None
    user_vector = (np.sum(weighted_vectors, axis=0) / np.sum(weights))
    norm = np.linalg.norm(user_vector)
    if norm == 0:
        return None
    user_vector = (user_vector / norm)
    return user_vector.astype(np.float32)

## 20. Test User Embedding

In [96]:
example_user = (consumer["consumer_id"].iloc[0])
user_vector = (build_user_embedding(example_user))
if user_vector is not None:
    print("User embedding shape:", user_vector.shape)
    print("User embedding norm:", np.linalg.norm(user_vector))

User embedding shape: (384,)
User embedding norm: 1.0


## 21. Personalized Semantic Retrieval

In [97]:
def semantic_recommend_for_user(user_id, top_k=10, retrieval_k=100):
    user_vector = (build_user_embedding(user_id))
    if user_vector is None:
        return []
    query_vector = ( user_vector.reshape(1, -1).astype(np.float32))
    scores, indices = (faiss_index.search(query_vector,retrieval_k))
    seen_items = set(consumer[consumer["consumer_id"].eq(user_id)]["item_id"])
    recommendations = []
    for score, idx in zip(scores[0],indices[0]):
        if idx == -1:
            continue
        item_id = (index_to_item[int(idx)])
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        recommendations.append({ "item_id": item_id,"semantic_score":float(score)})
        if len(recommendations) >= top_k:
            break
    result = pd.DataFrame(recommendations)
    if result.empty:
        return result
    result = result.merge(embedding_metadata[["item_id","title","language","item_type","producer_id"]],on="item_id",how="left")
    return result

## 22. Test Personalized Retrieval

In [98]:
semantic_recommendations = ( semantic_recommend_for_user(example_user,top_k=10,retrieval_k=100))
display(semantic_recommendations)

,item_id,semantic_score,title,language,item_type,producer_id
0,-4318248821095057296,0.877758,Gmail API: New endpoints for settings,en,HTML,-1387464358334758758
1,-1419188045265393093,0.569933,Google Cloud Endpoints now generally available...,en,HTML,3891637997717104548
2,1328618437884612347,0.534685,Google Cloud Machine Learning family grows wit...,en,HTML,3891637997717104548
3,-2424983931459616622,0.524548,All Together Now. Introducing G Suite.,en,HTML,-1387464358334758758
4,5206308811707978799,0.516812,The rise of APIs,en,HTML,-1032019229384696495
5,5619251370090681244,0.501210,Google Cloud Platform for AWS Professionals,en,HTML,3891637997717104548
6,8878031598946138458,0.498268,Inbox by Gmail now helps you track calendar ev...,en,HTML,-108842214936804958
7,-3850335246895770347,0.498195,You're Asking Too Much of Chat Bots. Just Let ...,en,HTML,-1443636648652872475
8,428486625959995206,0.496909,Most Interesting APIs in 2016: Cognitive Compu...,en,HTML,3609194402293569455
9,-2871347907044769386,0.489991,Making Sense of Unstructured Data with Google ...,en,HTML,-1443636648652872475


## 23. Validate Businees Rules

In [99]:
def validate_recommendations(user_id,recommendations):
    if recommendations.empty:
        return {"num_recommendations": 0,"seen_violations": 0,"availability_violations": 0,"language_violations": 0}
    recommendation_items = set(recommendations["item_id"])
    seen_items = set(consumer[consumer["consumer_id"].eq(user_id)]["item_id"])
    seen_violations = (recommendation_items&seen_items)
    unavailable = (recommendation_items - available_items)
    non_english = set(recommendations.loc[recommendations["language"].astype(str).str.lower().ne("en"),"item_id"])
    return {"num_recommendations": len(recommendation_items),"seen_violations": len(seen_violations), "availability_violations": len( unavailable), "language_violations": len( non_english)}

In [100]:
validation = validate_recommendations(example_user,semantic_recommendations)
print(validation)

{'num_recommendations': 10, 'seen_violations': 0, 'availability_violations': 0, 'language_violations': 0}


## 24. Handle Cold Start Users

In [101]:
def semantic_recommend_with_fallback(user_id,top_k=10):
    recommendations = (semantic_recommend_for_user(user_id,top_k=top_k,retrieval_k=100))
    if (recommendations is not None and len(recommendations) >= top_k):
        return recommendations
    seen_items = set(consumer[consumer["consumer_id"].eq(user_id)]["item_id"])
    fallback = []
    for item_id in global_popular_items:
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        if item_id not in english_items:
            continue
        fallback.append({"item_id":item_id,"semantic_score":0.0})
        if (len(fallback) + len(recommendations) >= top_k):
            break
    fallback_df = pd.DataFrame(fallback)
    if recommendations.empty:
        result = fallback_df
    else:
        result = pd.concat([recommendations,fallback_df],ignore_index=True)
    return result.head(top_k)

In [102]:
english_items = set(content.loc[content["language"].astype(str).str.lower().eq("en"),"item_id"])

## 25. Better Cold Start Fallback

In [103]:
def cold_start_recommend(user_id,k=10):
    seen = set(consumer[consumer["consumer_id"].eq(user_id)]["item_id"])
    recommendations = []
    for item_id in english_trending_items:
        if item_id in seen:
            continue
        recommendations.append(item_id)
        if len(recommendations) >= k:
            break
    return recommendations

## 26. Cliecked Article Similarity

In [104]:
def recommend_similar_news(item_id,user_id=None,top_k=10,retrieval_k=100):
    if item_id not in item_to_embedding_index:
        return pd.DataFrame()
    embedding_idx = (item_to_embedding_index[item_id])
    query_vector = (article_embeddings[embedding_idx].reshape(1, -1).astype(np.float32))
    scores, indices = (faiss_index.search(query_vector,retrieval_k))
    if user_id is not None:
        seen_items = set(consumer[consumer["consumer_id"].eq(user_id)]["item_id"])
    else:
        seen_items = set()
    results = []
    for score, idx in zip(scores[0],indices[0]):
        if idx == -1:
            continue
        candidate_item = (index_to_item[int(idx)])
        if candidate_item == item_id:
            continue
        if candidate_item in seen_items:
            continue
        if candidate_item not in available_items:
            continue
        results.append({"item_id": candidate_item,"semantic_score":float(score)})
        if len(results) >= top_k:
            break
    result = pd.DataFrame(results)
    if result.empty:
        return result
    return result.merge(embedding_metadata[["item_id","title","language","item_type","producer_id"]],on="item_id",how="left")

## 27. Test Clicked Article Recommendations

In [105]:
clicked_item = (consumer["item_id"].iloc[0])
similar_news = recommend_similar_news(item_id=clicked_item,user_id=example_user,top_k=10)
display(similar_news)

,item_id,semantic_score,title,language,item_type,producer_id
0,3510349622718287202,0.593798,20 Percent Will Stop Reading Your Email If You...,en,HTML,3609194402293569455
1,-1523691178084871,0.565370,Measuring email effectiveness in retail banking,en,HTML,5660542693104786364
2,8120046881003900118,0.522012,Email Isn't The Thing You're Bad At,en,HTML,-108842214936804958
3,-6659933088254630422,0.504253,How to E-Mail a Busy Person & get a Reply - De...,en,HTML,-5527145562136413747
4,5270696484536580646,0.446948,8 Insanely Simple Productivity Hacks,en,HTML,3609194402293569455
5,8878031598946138458,0.437890,Inbox by Gmail now helps you track calendar ev...,en,HTML,-108842214936804958
6,589254894302309232,0.428252,These 17 life hacks will change the way you us...,en,HTML,1895326251577378793
7,-5781461435447152359,0.411819,Inbox by Gmail: a better way to keep track of ...,en,HTML,-1032019229384696495
8,-4996336942690402156,0.409599,Live asynchronously.,en,HTML,-709287718034731589
9,8433131156569129937,0.403405,"Predicting the future, one week at a time",en,HTML,-108842214936804958


In [106]:
MIN_SEMANTIC_SIMILARITY = 0.30

In [107]:
SEMANTIC_MODELS = ["sentence-transformers/all-MiniLM-L6-v2", "sentence-transformers/all-mpnet-base-v2"]

## 28. Evaluate Semantic Retrieval

In [108]:
def evaluate_semantic_recommender(test_truth, k=10):
    precisions = []
    recalls = []
    hits = []
    ndcgs = []
    maps = []
    recommendation_catalog = set()
    for user_id, truth in test_truth.items():
        recommendations = (semantic_recommend_for_user(user_id,top_k=k,retrieval_k=100))
        if recommendations.empty:
            recommendation_items = []
        else:
            recommendation_items = (recommendations["item_id"].tolist())
        recommendation_catalog.update(recommendation_items)
        precisions.append(precision_at_k(recommendation_items,truth,k))
        recalls.append(recall_at_k(recommendation_items,truth,k))
        hits.append(hit_rate_at_k(recommendation_items,truth,k))
        ndcgs.append(ndcg_at_k(recommendation_items,truth,k))
        maps.append(average_precision_at_k(recommendation_items,truth,k))
    return {"Precision@10": np.mean(precisions), "Recall@10": np.mean(recalls), "HitRate@10": np.mean(hits), "NDCG@10":np.mean(ndcgs), "MAP@10":np.mean(maps), "Coverage@10":len(recommendation_catalog)}

In [109]:
def build_user_embedding(user_id, reference_time,history_df):
    history_df = consumer[(consumer["consumer_id"] == user_id) & (consumer["event_datetime"] < reference_time)]

## 29. Point In Time User Embedding

In [110]:
def build_user_embedding_at_time(user_id,cutoff_time,history_df):
    user_history = history_df[(history_df["consumer_id"] == user_id) & (history_df["event_datetime"] < cutoff_time)].sort_values("event_datetime")
    if user_history.empty:
        return None
    weighted_vectors = []
    weights = []
    for _, row in user_history.iterrows():
        item_id = row["item_id"]
        if item_id not in item_to_embedding_index:
            continue
        embedding_idx = (item_to_embedding_index[item_id])
        interaction_weight = float(row["interaction_weight"])
        recency_weight = (calculate_recency_weight(row["event_datetime"],cutoff_time))
        final_weight = (interaction_weight *  recency_weight)
        if final_weight <= 0:
            continue
        vector = (article_embeddings[embedding_idx])
        weighted_vectors.append(vector * final_weight)
        weights.append(final_weight)
    if not weighted_vectors:
        return None
    user_vector = (np.sum(weighted_vectors, axis=0) / np.sum(weights))
    norm = np.linalg.norm(user_vector)
    if norm == 0:
        return None
    user_vector = (user_vector / norm)
    return user_vector.astype(np.float32)

## 30. Point In Time Semantic Recomenders

In [111]:
def semantic_recommend_at_time(user_id,cutoff_time,history_df,top_k=10,retrieval_k=100):
    user_vector = (build_user_embedding_at_time(user_id,cutoff_time,history_df))
    if user_vector is None:
        return []
    query_vector = (user_vector.reshape(1, -1).astype(np.float32))
    scores, indices = (faiss_index.search(query_vector,retrieval_k))
    seen_items = set(history_df[(history_df["consumer_id"] == user_id) & (history_df["event_datetime"] < cutoff_time)]["item_id"])
    recommendations = []
    for score, idx in zip(scores[0],indices[0]):
        if idx == -1:
            continue
        item_id = (index_to_item[int(idx)])
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        recommendations.append(item_id)
        if len(recommendations) >= top_k:
            break
    return recommendations

## 31. Save User Embeddings

In [117]:
all_users = (consumer["consumer_id"].drop_duplicates().tolist())
user_embedding_rows = []
valid_user_ids = []
reference_time = (consumer["event_datetime"].max())
for user_id in all_users:
    vector = build_user_embedding(user_id=user_id, reference_time=reference_time, history_df=consumer)
    if vector is None:
        continue
    valid_user_ids.append(user_id)
    user_embedding_rows.append(vector)
if user_embedding_rows:
    user_embeddings = np.vstack(user_embedding_rows)
else:
    user_embeddings = np.empty((0,article_embeddings.shape[1]),dtype=np.float32)
print("User embedding matrix:", user_embeddings.shape)
print("Valid users:", len(valid_user_ids))

User embedding matrix: (0, 384)
Valid users: 0


In [118]:
np.save(FEATURE_DIR / "user_embeddings.npy", user_embeddings)
pd.DataFrame({"consumer_id": valid_user_ids, "embedding_index": np.arange(len(valid_user_ids))}).to_parquet(FEATURE_DIR / "user_embedding_index.parquet",index=False)

In [120]:
if len(user_embeddings) > 0:
    user_faiss_index = (faiss.IndexFlatIP(user_embeddings.shape[1]))
    user_faiss_index.add(user_embeddings.astype(np.float32))
    faiss.write_index(user_faiss_index,str(INDEX_DIR / "user_interest.index"))

## 32. Semantic Score As A Ranking Features

In [128]:
def generate_semantic_candidates(user_id,retrieval_k=100):
    user_vector = (build_user_embedding(user_id))
    if user_vector is None:
        return pd.DataFrame(columns=["consumer_id","item_id","semantic_score"])
    query_vector = (user_vector.reshape(1, -1).astype(np.float32))
    scores, indices = (faiss_index.search(query_vector,retrieval_k))
    seen_items = get_seen_items(user_id)
    rows = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item_id = (index_to_item[int(idx)])
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"semantic_score": float(score)})
    return pd.DataFrame(rows)

## 33. Save Semantic Candidate Dataset

In [129]:
consumer = consumer.sort_values(["consumer_id", "event_datetime"]).reset_index(drop=True)
test_truth = (consumer.groupby("consumer_id").tail(1).groupby("consumer_id")["item_id"].apply(set).to_dict())
train_consumer = consumer.loc[~consumer.index.isin(consumer.groupby("consumer_id").tail(1).index)].copy()
print("Users in test set:", len(test_truth))
print("Training interactions:", len(train_consumer))
print("Test interactions:", sum(len(v) for v in test_truth.values()))

Users in test set: 1895
Training interactions: 70417
Test interactions: 1895


In [130]:
def generate_semantic_candidates(user_id,history_df,retrieval_k=100):
    user_vector = build_user_embedding(user_id=user_id,reference_time=history_df["event_datetime"].max(),history_df=history_df)
    if user_vector is None:
        return pd.DataFrame(columns=["consumer_id","item_id","semantic_score"])
    scores, indices = faiss_index.search(user_vector.reshape(1, -1).astype(np.float32),retrieval_k)
    rows = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        item_id = index_to_item[int(idx)]
        rows.append({"consumer_id": user_id,"item_id": item_id,"semantic_score": float(score)})
    candidates = pd.DataFrame(rows)
    if candidates.empty:
        return candidates
    seen_items = set(history_df.loc[history_df["consumer_id"].eq(user_id), "item_id"])
    candidates = candidates[~candidates["item_id"].isin(seen_items)]
    candidates = candidates[candidates["item_id"].isin(available_items)]
    candidates = candidates[candidates["item_id"].isin(set(semantic_articles["item_id"]))]
    return candidates.reset_index(drop=True)

In [131]:
candidate_frames = []
for user_id in test_truth.keys():
    candidates = generate_semantic_candidates(user_id=user_id,history_df=train_consumer,retrieval_k=100)
    if not candidates.empty:
        candidate_frames.append(candidates)
if candidate_frames:
    semantic_candidates = pd.concat(candidate_frames,ignore_index=True)
else:
    semantic_candidates = pd.DataFrame(columns=["consumer_id","item_id","semantic_score"])
semantic_candidates.to_parquet(SEMANTIC_DIR / "semantic_candidates.parquet",index=False)
print("Semantic candidates:", semantic_candidates.shape)
print("Unique users:",semantic_candidates["consumer_id"].nunique())
print("Unique articles:",semantic_candidates["item_id"].nunique())

Semantic candidates: (0, 3)
Unique users: 0
Unique articles: 0


In [132]:
user_interaction_counts = (consumer.groupby("consumer_id").size())
eligible_users = user_interaction_counts[user_interaction_counts >= 2].index
test_truth = (consumer[consumer["consumer_id"].isin(eligible_users)].groupby("consumer_id").tail(1).groupby("consumer_id")["item_id"].apply(set).to_dict())
test_indices = (consumer[consumer["consumer_id"].isin(eligible_users)].groupby("consumer_id").tail(1).index)
train_consumer = consumer.loc[~consumer.index.isin(test_indices)].copy()
print("Eligible users:", len(test_truth))
print("Train interactions:", len(train_consumer))
print("Test interactions:", sum(len(x) for x in test_truth.values()))

Eligible users: 1715
Train interactions: 70597
Test interactions: 1715


## 34. Create Semantic Recommendation Reports

In [134]:
semantic_report = (semantic_candidates.merge(embedding_metadata[["item_id","title","language","item_type","producer_id"]], on="item_id", how="left"))
semantic_report = (semantic_report.sort_values(["consumer_id","semantic_score"],ascending=[True,False]))
semantic_report.to_csv(SEMANTIC_DIR /"semantic_recommendations.csv",index=False)

## 35. Measure Semantic Retrieval Coverage

In [136]:
semantic_coverage = (semantic_candidates["item_id"].nunique())
total_available_english = len(available_items & english_items)
coverage_percentage = (semantic_coverage / max(total_available_english, 1) * 100)
print("Semantic catalog coverage:", semantic_coverage)
print("Coverage percentage:", round(coverage_percentage, 2))

Semantic catalog coverage: 0
Coverage percentage: 0.0


## 36. Validate Duplicates Recommendations

In [137]:
duplicate_count = (semantic_candidates.duplicated(["consumer_id","item_id"]).sum())
print("Duplicate user-item candidates:",duplicate_count)

Duplicate user-item candidates: 0


## Save Semantic Configurations

In [139]:
semantic_config = {"embedding_model": EMBEDDING_MODEL_NAME, "embedding_dimension": int(article_embeddings.shape[1]), "retrieval_k": RETRIEVAL_K, "top_k": TOP_K, "recency_half_life_days": RECENCY_HALF_LIFE_DAYS, "english_only": True, "available_only":   True, "normalized_embeddings":  True, "faiss_metric": "inner_product_equivalent_to_cosine"}

import json
with open(SEMANTIC_DIR / "semantic_config.json", "w") as f:
    json.dump(semantic_config,f,indent=4)